# Model Playground

The first-class local dev loop for **scale-forecasting**: pick any registered model, run it on a small sample dataset, and see the forecast + metric panel — all offline, no GCP.

It runs the **same code path** as a production run (`playground` → `worker.run_cell`), so what you see here is what runs at scale. To add your own model, copy `docs/model_template.py` into `src/scale_forecasting/models/` and re-run the cells — it appears in the dropdown automatically.

Read order if you're new: `README.md` → `config.py` → `worker.py` → one file under `models/`.

## 1. Get the code (cloud runtimes only)

On a cloud notebook (Colab Enterprise, Vertex Workbench) this cell clones or updates the repo so you're always on the latest `src/`. **Skip it when running inside a local clone** — it's a no-op guarded on the repo already being importable.

In [ ]:
# Cloud bootstrap: clone the repo and install it (editable) so `import scale_forecasting`
# resolves. Harmless locally — if the package already imports, we do nothing.
import importlib.util, os, subprocess, sys

REPO_URL = os.environ.get("SF_REPO_URL", "https://github.com/statmike/scale-forecasting.git")
REPO_DIR = os.environ.get("SF_REPO_DIR", "scale-forecasting")

if importlib.util.find_spec("scale_forecasting") is None:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    print("installed scale_forecasting from", REPO_DIR)
else:
    print("scale_forecasting already importable — skipping bootstrap")

## 2. Discover the models

`available_models()` reads the factory registry, so every model file that ends with `register(...)` shows up here — including one you just added.

In [ ]:
from scale_forecasting.playground import available_models, run_model, sample_data, summarize

MODELS = available_models()
print("available models:")
for m in MODELS:
    print(" ", m)

## 3. Look at the sample data

A small, deterministic panel from the real generator (same code as the shipped 100k dataset). Five archetypes make different models win — pick a `ts_id` to focus on.

In [ ]:
data = sample_data(n_series=5, history=730, freq="D")
print(data.groupby("ts_id")["archetype"].first())
data.head()

In [ ]:
# Plot the history of each sample series.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
for ts_id, g in data.groupby("ts_id"):
    ax.plot(g["ds"], g["y"], label=f"{ts_id} ({g['archetype'].iloc[0]})", lw=1)
ax.legend(fontsize=8, ncol=2)
ax.set_title("sample series")
plt.show()

## 4. Run a model

Change `MODEL` and `TS_ID` and re-run. `backtest=True` scores the model (a 3-fold time-series CV) so the metric panel is populated; with it off, metrics are NaN by design (a single full-fit run doesn't score itself). This calls the real `worker.run_cell` — a model failure comes back as an *error cell*, it never raises.

In [ ]:
MODEL = "theta"      # any name from section 2
TS_ID = "s_000000"   # any series from section 3
HORIZON = 28

run = run_model(MODEL, data=data, ts_id=TS_ID, horizon=HORIZON, backtest=True)
print(summarize(run))

## 5. The prediction frame + metric panel

Every model returns the identical canonical frame (`ds, yhat, yhat_lower, yhat_upper, quantiles`) in original units, with ordered bounds.

In [ ]:
import pandas as pd

print("metric panel (backtest):")
print(pd.Series(run.result.metrics).round(4))
print("\nforecast frame (head):")
run.result.predictions.head()

In [ ]:
# Plot history + forecast with the prediction interval.
hist = run.series
pred = run.result.predictions

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hist["ds"], hist["y"], label="history", lw=1, color="#444")
ax.plot(pred["ds"], pred["yhat"], label="forecast", lw=2, color="#1f77b4")
ax.fill_between(
    pred["ds"], pred["yhat_lower"], pred["yhat_upper"],
    alpha=0.2, color="#1f77b4", label="interval",
)
ax.legend()
ax.set_title(f"{MODEL} — {TS_ID}")
plt.show()

## 6. Add your own model

1. Copy `docs/model_template.py` → `src/scale_forecasting/models/my_model.py`.
2. Rename the class, set `name = "my_model"`, fill in `fit` / `predict`.
3. Add `my_model,` to the import block in `src/scale_forecasting/models/__init__.py`.
4. Restart the kernel and re-run section 2 — `my_model` is now in the list.

Full walkthrough and the model contract: `docs/adding_a_model.md`.